# Extra Credit: Neural Network on a UCI Dataset

You've seen the template work on the Parkinson's data. Now do the same thing on a dataset **you** pick from the UCI Machine Learning Repository: https://archive.ics.uci.edu/

## What to do

1. Pick a UCI dataset that interests you.
2. Tell me what you picked and why (Step 1 below).
3. Load it, explore it, and preprocess it.
4. Train **at least three** different neural network architectures on it.
5. Compare them and write up the results of the best one.

## Constraints on the dataset

- **Tabular data only** — rows × columns of numbers and/or categories. Skip image, audio, and raw text datasets; those need CNNs / RNNs we haven't covered.
- **At least ~300 samples and 5 features.** Smaller than that and the NN won't have enough to chew on.
- **Either a regression target (numeric) or a classification target (a class label).** The template defaults to regression — if you pick classification, you'll swap `MLPRegressor` for `MLPClassifier` and change your metrics. There's a note in the code where to do that.

Tip on browsing UCI: on the dataset list page, the left-hand filters let you narrow by **Task** (Classification / Regression), **Feature Type** (Numerical / Categorical / Mixed), and **# of Instances**. That's the fastest way to find something that fits the constraints above.

## Grading (extra credit points)

| Points | What I'm looking for |
|---|---|
| 3 | Clear justification of dataset choice + why a NN is reasonable for it |
| 2 | Clean data loading and preprocessing (handles missing values, encodes categoricals, scales features) |
| 3 | Three thoughtful architectures — different shapes with reasoning, not three random guesses |
| 3 | Clear comparison + analysis of why the best one won |
| 1 | Code runs end-to-end without errors |


## Step 1: Pick your dataset

Fill out the box below **before** you start coding. Half the value of this assignment is in justifying the choice.

> **Dataset name:** _________________
>
> **UCI ID number:** _____ &nbsp;&nbsp;(find this on the dataset's UCI page — it's in the URL, e.g. `archive.ics.uci.edu/dataset/189/...` → id is 189)
>
> **Link to the dataset page:** _________________
>
> **Problem type:** Regression  /  Classification  *(delete one)*
>
> **Number of samples / features:** _____ rows × _____ columns
>
> **What is the target?** What are you trying to predict?
>
> _________________
>
> **Why this dataset?** What about it grabbed you?
>
> _________________
>
> **Why is a neural network a reasonable tool for this?**  *(Hint: NNs shine when relationships are nonlinear and features interact in tangled ways. They're overkill for clean linear problems.)*
>
> _________________


In [ ]:
# Setup — same imports as the in-class lab.
# IF YOU PICKED CLASSIFICATION: uncomment the MLPClassifier and accuracy_score lines,
# and comment out MLPRegressor / r2_score / mean_absolute_error.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# from sklearn.neural_network import MLPClassifier
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load YOUR dataset.
# Easiest path: the ucimlrepo package — works for most UCI datasets.
# Just put your dataset's UCI id below.

from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=___)        # <-- put your UCI id number here

features = dataset.data.features.copy()
targets  = dataset.data.targets.copy()

# Combine into one frame so you can see the whole thing
data = pd.concat([features, targets], axis=1)

print(f"Shape:   {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()


## Step 2: Explore the data

Before you scale and split, look for problems that will hurt the model:

- **Missing values** (NaNs)
- **Categorical columns** that need encoding (`object` or `category` dtypes)
- **Wildly different scales** between columns — scaling will fix this, but flag if anything looks weird (negative ages, etc.)


In [ ]:
print("Missing values per column:")
print(data.isna().sum())

print("\nData types:")
print(data.dtypes)

print("\nNumeric summary:")
data.describe()


## Step 3: Preprocess

Handle whatever you found above. Common moves:

- Drop missing rows: `data = data.dropna()`
- Or fill them: `data = data.fillna(data.mean(numeric_only=True))`
- One-hot encode categoricals: `X = pd.get_dummies(X, drop_first=True)`
- Pick your target column and split features from target


In [ ]:
# Drop or fill missing values if needed
# data = data.dropna()

# Pick your target column — replace with the actual column name
target_col = '____'                    # <-- put your target column name here

X = data.drop(columns=[target_col])
y = data[target_col]

# One-hot encode any categorical features (no-op if everything is numeric)
X = pd.get_dummies(X, drop_first=True)

print(f"Features X: {X.shape[1]} columns, {X.shape[0]} rows")
print(f"Target y:   {target_col}")

# Train/test split — same recipe as the in-class lab
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features — REQUIRED for neural networks
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTraining set: {X_train.shape[0]} examples")
print(f"Test set:     {X_test.shape[0]} examples")


## Reminder: early stopping is on for every architecture

Same three parameters as the in-class lab:

```python
early_stopping=True       # hold out a validation chunk and watch it
validation_fraction=0.15  # 15% of training data goes to validation
n_iter_no_change=20       # stop if validation hasn't improved in 20 iterations
```

After fitting, also look at `model.n_iter_` (how many iterations actually ran). A bigger architecture that stops earlier is often a sign of overfitting — early stopping caught it in the act.


## Step 4: Architecture 1

**Your architecture (hidden layer sizes):** _________________

**Why this shape?** _________________


In [ ]:
model1 = MLPRegressor(
    hidden_layer_sizes=(?,),         # <-- your shape here, e.g. (16,) or (32, 16)
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)
model1.fit(X_train_scaled, y_train)
predictions1 = model1.predict(X_test_scaled)

r2_1  = r2_score(y_test, predictions1)
mae_1 = mean_absolute_error(y_test, predictions1)

print(f"Architecture 1:")
print(f"  R-squared:                 {r2_1:.4f}")
print(f"  MAE:                       {mae_1:.2f}")
print(f"  Stopped at iteration:      {model1.n_iter_} of {model1.max_iter}")
print(f"  Best validation R-squared: {model1.best_validation_score_:.4f}")

# CLASSIFICATION? Use this block instead of r2/MAE:
# acc_1 = accuracy_score(y_test, predictions1)
# print(f"  Accuracy: {acc_1:.4f}")


## Step 5: Architecture 2

**Your architecture (hidden layer sizes):** _________________

**Why this shape?** _________________


In [ ]:
model2 = MLPRegressor(
    hidden_layer_sizes=(?,),         # <-- your shape here, e.g. (16,) or (32, 16)
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)
model2.fit(X_train_scaled, y_train)
predictions2 = model2.predict(X_test_scaled)

r2_2  = r2_score(y_test, predictions2)
mae_2 = mean_absolute_error(y_test, predictions2)

print(f"Architecture 2:")
print(f"  R-squared:                 {r2_2:.4f}")
print(f"  MAE:                       {mae_2:.2f}")
print(f"  Stopped at iteration:      {model2.n_iter_} of {model2.max_iter}")
print(f"  Best validation R-squared: {model2.best_validation_score_:.4f}")

# CLASSIFICATION? Use this block instead of r2/MAE:
# acc_2 = accuracy_score(y_test, predictions2)
# print(f"  Accuracy: {acc_2:.4f}")


## Step 6: Architecture 3

**Your architecture (hidden layer sizes):** _________________

**Why this shape?** _________________


In [ ]:
model3 = MLPRegressor(
    hidden_layer_sizes=(?,),         # <-- your shape here, e.g. (16,) or (32, 16)
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)
model3.fit(X_train_scaled, y_train)
predictions3 = model3.predict(X_test_scaled)

r2_3  = r2_score(y_test, predictions3)
mae_3 = mean_absolute_error(y_test, predictions3)

print(f"Architecture 3:")
print(f"  R-squared:                 {r2_3:.4f}")
print(f"  MAE:                       {mae_3:.2f}")
print(f"  Stopped at iteration:      {model3.n_iter_} of {model3.max_iter}")
print(f"  Best validation R-squared: {model3.best_validation_score_:.4f}")

# CLASSIFICATION? Use this block instead of r2/MAE:
# acc_3 = accuracy_score(y_test, predictions3)
# print(f"  Accuracy: {acc_3:.4f}")


## Step 7: Compare your three architectures


In [ ]:
# Side-by-side comparison
print("\n" + "="*64)
print("ARCHITECTURE COMPARISON")
print("="*64)
print(f"{'Architecture':<28} {'R-squared':<12} {'MAE':<10} {'n_iter':<8}")
print("-"*64)
print(f"{'1. (your shape)':<28} {r2_1:<12.4f} {mae_1:<10.2f} {model1.n_iter_:<8}")
print(f"{'2. (your shape)':<28} {r2_2:<12.4f} {mae_2:<10.2f} {model2.n_iter_:<8}")
print(f"{'3. (your shape)':<28} {r2_3:<12.4f} {mae_3:<10.2f} {model3.n_iter_:<8}")
print("="*64)

scores = [
    ('Architecture 1', r2_1),
    ('Architecture 2', r2_2),
    ('Architecture 3', r2_3),
]
best_name, best_score = max(scores, key=lambda x: x[1])
print(f"\nBest performer: {best_name} with R-squared = {best_score:.4f}")


## Step 8: Best model — describe the results

Answer all five. **Don't just write "Architecture 2 won because R² was 0.71."** Go deeper than the number.

1. **Which architecture won, and what does its R² (or accuracy) actually mean for this dataset?**
   Translate the metric back into the real world. *Example: "R² of 0.71 means the model explains 71% of the variation in the target — for predicting house prices, that's decent but not great."*

   _Your answer:_

2. **Look at the MAE (or for classification, the confusion matrix). What's the typical error?**
   Is that error small enough to be useful for whoever would use this model in real life?

   _Your answer:_

3. **Where does the winning architecture sit on the shape spectrum — narrow, wide, deep, shallow?**
   Why might that shape have been the right fit for *this particular* dataset?

   _Your answer:_

4. **Compare it to the architecture that did the worst. Why did the worst one struggle?**
   Underfitting (not enough capacity)? Overfitting (too much, validation stopped it early)? Something else?

   _Your answer:_

5. **What would you try next if you had more time?**
   A different shape? More iterations? More data? A different model entirely?

   _Your answer:_
